# 将实验结果导出为 LaTeX 表格
把本 Notebook 放入项目的 `notebook/`，在 VS Code 选择原实验的 Python 内核，依次运行所有单元格。代码从仓库的 `results/` 读取已导出的实验数据，将表格写入仓库根目录的 `tables/`。需要 pandas。

| 输入 CSV（均位于 results/） | 生成内容 |
|---|---|
| validation_model_comparison_2025.csv | 验证集模型比较 |
| validation_holiday_comparison_2025.csv | 验证集节假日特征对照 |
| validation_peak_comparison_2025.csv | 验证集高峰日与非高峰日误差 |
| test_metrics_2026.csv | 测试集总体误差、分组误差 |
| peak_underprediction_cases_2026.csv | 低估最大的五天 |
| peak_case_traveller_composition_2026.csv | 案例日期的香港居民入境人次、参照值及占比 |

逐日预测文件用于复核与画图。旅客构成 CSV 保留了三类旅客的完整数据，正文导出的表聚焦香港居民。

## 1. 找到项目目录并读取结果

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

current = Path.cwd()
project_root = next(
    (folder for folder in (current, *current.parents)
     if (folder / "results").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError("请在 VS Code 打开项目根目录，并从仓库内运行本 Notebook。")

results_dir = project_root / "results"
tables_dir = project_root / "tables"
names = [
    "validation_model_comparison_2025",
    "validation_holiday_comparison_2025",
    "validation_peak_comparison_2025",
    "test_metrics_2026",
    "peak_underprediction_cases_2026",
    "peak_case_traveller_composition_2026",
]
missing = [name for name in names if not (results_dir / f"{name}.csv").is_file()]
if missing:
    raise FileNotFoundError("results/ 缺少：" + "、".join(f"{name}.csv" for name in missing))

results = {
    name: pd.read_csv(results_dir / f"{name}.csv", encoding="utf-8-sig")
    for name in names
}
print("读取目录", results_dir)
display(pd.DataFrame({"CSV": [f"{name}.csv" for name in names],
                      "行数": [len(results[name]) for name in names]}))

读取目录 d:\course\ds and ai\shenzhen-bay-passenger-forecast\results


,CSV,行数
0,validation_model_comparison_2025.csv,4
1,validation_holiday_comparison_2025.csv,3
2,validation_peak_comparison_2025.csv,10
3,test_metrics_2026.csv,12
4,peak_underprediction_cases_2026.csv,5
5,peak_case_traveller_composition_2026.csv,15


## 2. 整理验证集与测试集
按 CSV 的日期类型区分总体和分组结果。节假日对照保留加入前后的 MAE、RMSE，百分比列的数值直接来自原表。

In [2]:
def report_columns(frame, columns):
    return frame.loc[:, list(columns)].rename(columns=columns)

basic_columns = {"model": "模型", "MAE": "MAE", "RMSE": "RMSE"}
group_columns = {
    "day_type": "日期类型", "model": "模型", "days": "天数",
    "MAE": "MAE", "RMSE": "RMSE", "underprediction_rate_pct": "低估率（%）",
}
day_order = {"高峰日": 0, "非高峰日": 1}

def group_report(frame):
    groups = frame.loc[frame["day_type"].isin(day_order)].copy()
    groups["day_order"] = groups["day_type"].map(day_order)
    return report_columns(groups.sort_values(["day_order", "MAE"]), group_columns)

test_metrics = results["test_metrics_2026"]
tables = {
    "validation_models_2025": report_columns(
        results["validation_model_comparison_2025"].sort_values("MAE"), basic_columns
    ),
    "validation_holidays_2025": report_columns(
        results["validation_holiday_comparison_2025"],
        {"model": "模型", "MAE_without_holidays": "无节假日 MAE",
         "MAE_with_holidays": "含节假日 MAE", "RMSE_without_holidays": "无节假日 RMSE",
         "RMSE_with_holidays": "含节假日 RMSE", "MAE_reduction_pct": "MAE 降幅（%）"},
    ),
    "validation_peak_2025": group_report(results["validation_peak_comparison_2025"]),
    "test_overall_2026": report_columns(
        test_metrics.loc[test_metrics["day_type"].eq("全部日期")].sort_values("MAE"),
        basic_columns,
    ),
    "test_by_day_type_2026": group_report(test_metrics),
}
display(tables["test_overall_2026"])

,模型,MAE,RMSE
3,Ridge (with holidays),6465.037528,9443.724348
9,Gradient Boosting (with holidays),6736.407164,10170.157195
6,Ridge (without holidays),7218.694188,11430.797381
0,Last week's same day,10874.818482,18407.861353


## 3. 整理高峰日案例
按含节假日 Ridge 的低估人次由大到小排列。香港居民参照值为此前四周同星期的中位数，表中占比以当天总入境客流为分母。

In [3]:
cases = results["peak_underprediction_cases_2026"].sort_values(
    "shortfall_with_holidays", ascending=False
)
tables["peak_underprediction_2026"] = report_columns(cases, {
    "date": "日期", "actual_arrivals": "实际入境",
    "shortfall_without_holidays": "Ridge 无节假日低估",
    "shortfall_with_holidays": "Ridge 含节假日低估",
})
composition = results["peak_case_traveller_composition_2026"]
residents = composition.loc[composition["旅客类别"].eq("香港居民")]
residents = residents.set_index("日期").loc[cases["date"]].reset_index()
tables["peak_hk_residents_2026"] = report_columns(residents, {
    "日期": "日期", "当天入境人次": "香港居民入境",
    "此前四周同星期中位数": "参照中位数",
    "增加人次": "增加人次", "占当天总客流（%）": "占总客流（%）",
})
display(tables["peak_hk_residents_2026"])

,日期,香港居民入境,参照中位数,增加人次,占总客流（%）
0,2026-05-25,124251,46677.5,77573.5,90.282289
1,2026-10-19,121513,49597.5,71915.5,86.731191
2,2026-04-06,112557,47110.0,65447.0,89.361449
3,2026-09-27,103552,92962.5,10589.5,83.844379
4,2026-08-09,89255,79250.5,10004.5,75.435260


## 4. 导出表格并复制 Overleaf 插入代码
上传生成的 `tables/` 到 Overleaf 根目录。`main.tex` 需加载 `booktabs`（生成横线）和 `graphicx`（限制宽表宽度）。下方输出的插入代码包含标题和标签，复制所需表格的代码到对应章节。

In [4]:
model_names = {
    "Last week's same day": "上周同日基准",
    "Ridge": "Ridge",
    "Random Forest": "随机森林",
    "Gradient Boosting": "梯度提升",
    "Ridge (with holidays)": "Ridge（含节假日）",
    "Ridge (without holidays)": "Ridge（无节假日）",
    "Random Forest (with holidays)": "随机森林（含节假日）",
    "Gradient Boosting (with holidays)": "梯度提升（含节假日）",
}
captions = {
    "validation_models_2025": "2025 年验证集模型比较（误差单位：人次）",
    "validation_holidays_2025": "2025 年验证集节假日特征对照（误差单位：人次）",
    "validation_peak_2025": "2025 年验证集分日期类型预测误差（误差单位：人次）",
    "test_overall_2026": "2026 年测试集总体预测误差（单位：人次）",
    "test_by_day_type_2026": "2026 年测试集分日期类型预测误差（误差单位：人次）",
    "peak_underprediction_2026": "2026 年测试集典型低估日期（单位：人次）",
    "peak_hk_residents_2026": "典型低估日期的香港居民入境人次及占比",
}

latex_escapes = {
    "\\": r"\textbackslash{}", "&": r"\&", "%": r"\%", "$": r"\$",
    "#": r"\#", "_": r"\_", "{": r"\{", "}": r"\}",
    "~": r"\textasciitilde{}", "^": r"\textasciicircum{}",
}

def latex_cell(value):
    if pd.isna(value):
        return "--"
    if isinstance(value, float):
        return f"{value:,.2f}"
    return "".join(latex_escapes.get(char, char) for char in str(value))

def latex_tabular(table, alignment):
    rows = [table.columns, *table.itertuples(index=False, name=None)]
    lines = [rf"\begin{{tabular}}{{{alignment}}}", r"\toprule"]
    for index, row in enumerate(rows):
        lines.append(" & ".join(latex_cell(value) for value in row) + r" \\")
        if index == 0:
            lines.append(r"\midrule")
    lines.extend([r"\bottomrule", r"\end{tabular}"])
    return "\n".join(lines) + "\n"

tables_dir.mkdir(exist_ok=True)
for name, table in tables.items():
    if table.empty:
        raise ValueError(f"{name} 无数据，请核对 CSV 的筛选字段。")
    table = table.copy()
    if "模型" in table:
        table["模型"] = table["模型"].replace(model_names)
    alignment = "".join(
        "r" if pd.api.types.is_numeric_dtype(table[column]) else "l"
        for column in table
    )
    latex = latex_tabular(table, alignment)
    (tables_dir / f"{name}.tex").write_text(latex, encoding="utf-8")

print(f"已导出 {len(tables)} 张表到 {tables_dir}\n")
for name, caption in captions.items():
    print(r"\begin{table}[htbp]")
    print(r"    \centering")
    print(rf"    \caption{{{caption}}}")
    print(rf"    \label{{tab:{name.replace('_', '-')}}}")
    if name == "validation_holidays_2025":
        print(rf"    \resizebox{{\linewidth}}{{!}}{{\input{{tables/{name}}}}}")
    else:
        print(rf"    \input{{tables/{name}}}")
    print(r"\end{table}", end="\n\n")

已导出 7 张表到 d:\course\ds and ai\shenzhen-bay-passenger-forecast\tables

\begin{table}[htbp]
    \centering
    \caption{2025 年验证集模型比较（误差单位：人次）}
    \label{tab:validation-models-2025}
    \input{tables/validation_models_2025}
\end{table}

\begin{table}[htbp]
    \centering
    \caption{2025 年验证集节假日特征对照（误差单位：人次）}
    \label{tab:validation-holidays-2025}
    \resizebox{\linewidth}{!}{\input{tables/validation_holidays_2025}}
\end{table}

\begin{table}[htbp]
    \centering
    \caption{2025 年验证集分日期类型预测误差（误差单位：人次）}
    \label{tab:validation-peak-2025}
    \input{tables/validation_peak_2025}
\end{table}

\begin{table}[htbp]
    \centering
    \caption{2026 年测试集总体预测误差（单位：人次）}
    \label{tab:test-overall-2026}
    \input{tables/test_overall_2026}
\end{table}

\begin{table}[htbp]
    \centering
    \caption{2026 年测试集分日期类型预测误差（误差单位：人次）}
    \label{tab:test-by-day-type-2026}
    \input{tables/test_by_day_type_2026}
\end{table}

\begin{table}[htbp]
    \centering
    \caption{2026 年测试集典型低估日期（单位：人次）}
